# 06 — Embedding drift without retraining the LLM

Vendor embedding APIs change when the provider ships a new encoder. You cannot "retrain" that model inside your account. Practitioners instead:

1. **Pin** the provider model ID in config.
2. Keep a **reference batch** of embeddings (hashed IDs, not necessarily raw text).
3. Track **cosine distance** or retrieval hit-rate degradation.
4. Refresh indexes / RAG corpora and run **eval gates** before traffic moves.

This notebook simulates a provider swap with normalized random vectors—no GPU LLM required.


In [ ]:
# From repo root: pip install -e ".[dev]"
%matplotlib inline

import numpy as np
import pandas as pd

from drift_lab.viz import embedding_distance_figure

rng = np.random.default_rng(7)
n_batches = 60
dim = 32
n_refs = 200

ref = rng.normal(size=(n_refs, dim))
ref /= np.linalg.norm(ref, axis=1, keepdims=True)

distances = []
for t in range(n_batches):
  # provider noise increases after batch 25 (synthetic "model ID bump")
    drift_scale = 0.02 * max(0, t - 25)
    batch = ref + rng.normal(scale=0.05 + drift_scale, size=ref.shape)
    batch /= np.linalg.norm(batch, axis=1, keepdims=True)
    cos_dist = 1.0 - (batch @ ref.T).max(axis=1)
    distances.append(float(cos_dist.mean()))

dist_arr = np.array(distances)
print("Batches:", len(dist_arr), "| dim:", dim)


## Mean cosine distance to reference batch


In [ ]:
fig, meta = embedding_distance_figure(dist_arr)
meta


## When to act (example thresholds on surrogate distance)


In [ ]:
THRESHOLD = 0.08
breach = int(np.argmax(dist_arr > THRESHOLD)) if (dist_arr > THRESHOLD).any() else None
print("First batch above", THRESHOLD, ":", breach)
print("Action: pin model ID in CI, refresh vector index, run pinned eval set before traffic switch")


## Governance levers (tabular vs embedding vs generative)


In [ ]:
pd.DataFrame(
    [
        ("Tabular head", "Retrain / reweight on fresh labels", "Rolling accuracy, PSI"),
        ("Embedding API", "Pin model ID; refresh vector index", "Reference-batch cosine distance"),
        ("Prompt + RAG", "Hash prompts; version corpus", "Pinned eval set regression"),
    ],
    columns=["layer", "maintenance action", "monitor"],
)


## Takeaway

Full weight retrain is one tool among many. Match the **maintenance action to the layer** that actually moved.
